# Import

In [1]:
import string
import IPython
from IPython.display import Audio
import torch
import os

import torchaudio

from TTS.tts.utils.synthesis import synthesis
try:
  from TTS.utils.audio import AudioProcessor
except:
  from TTS.utils.audio import AudioProcessor
from TTS.tts.models import setup_model
from TTS.config import load_config
from TTS.tts.models.vits import *
from TTS.tts.utils.speakers import SpeakerManager
from TTS.utils.vad import get_vad_model_and_utils, remove_silence, resample_wav, read_audio

from pydub import AudioSegment
import numpy as np

from transcriber import process_audio

# Define Parameter and Constant

In [2]:
OUT_PATH = 'output/pipeline'
BASE_MODEL_PATH = './model/base_th_en'
REFERENCE_FILENAME = 'Tsync2_905_mic1.flac'

# model vars 
MODEL_PATH = os.path.join(BASE_MODEL_PATH, 'checkpoint_413001.pth')
CONFIG_PATH = os.path.join(BASE_MODEL_PATH, 'config.json')
TTS_LANGUAGES = os.path.join(BASE_MODEL_PATH, 'language_ids.json')
USE_CUDA = torch.cuda.is_available()
REFERENCE_PATH = os.path.join("./reference_voice", REFERENCE_FILENAME.strip().strip('./').strip('/'))
REFERENCE_WAV_PATH = os.path.join("./reference_voice", "wav", REFERENCE_FILENAME.strip().strip('./').strip('/').split(".")[0] + ".wav")
SPEAKER_FOLDER = REFERENCE_PATH.split("/")[-1].split(".")[0]

model_name = MODEL_PATH.rstrip('.pth').split('/')[-1]

# Get translation

In [3]:
# Get GEMINI_API_KEY
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY is not set. Please set the GEMINI_API_KEY environment variable at .env file.")

# Call API to Gemini
transcribe, translation = process_audio(REFERENCE_PATH, GEMINI_API_KEY, keep_transliterate_english = True)

Sending request to Gemini API...


In [4]:
transcribe

'ความจริงโดยเฉพาะย่านหัวหมากและสีลม พบว่ามีโต๊ะรับแทงรายย่อยนับร้อยโต๊ะที่ส่งยอดพนันบอลผ่านโต๊ะย่อยอีกสองชั้น ไปถึงโต๊ะพนันบอลใหญ่ที่จับกุมได้'

In [5]:
translation = "เมื่อเดือนที่แล้ว ฉันได้มีโอกาสไปร่วมงานสัมมนา Digital Transformation ที่จัดโดย Microsoft ณ True Digital Park ซึ่งมีวิทยากรจากต่างประเทศมาแชร์ประสบการณ์เกี่ยวกับการใช้ AI และ Cloud Computing ในการพัฒนาธุรกิจยุคใหม่"

In [6]:
if transcribe and translation:
    transcribe = transcribe.strip('"')
    translation = translation.translate(str.maketrans('', '', string.punctuation)).strip('\n')

# Setup Model and Config

In [7]:
# load the config
C = load_config(CONFIG_PATH)

# load the audio processor
ap = AudioProcessor(**C.audio)

# override config
C["speakers_file"] = None
C["d_vector_file"] = []
C["language_ids_file"] = TTS_LANGUAGES

C["model_args"]["speakers_file"] = None
C["model_args"]["d_vector_file"] = []
C["model_args"]["language_ids_file"] = TTS_LANGUAGES

C.model_args['use_speaker_encoder_as_loss'] = False

model = setup_model(C)
cp = torch.load(MODEL_PATH, map_location=torch.device('cpu'))

# remove speaker encoder
model_weights = cp['model'].copy()
for key in list(model_weights.keys()):
  if "speaker_encoder" in key:
    del model_weights[key]

model.load_state_dict(model_weights)

model.eval()

if USE_CUDA:
  model = model.cuda()

# synthesize voice
use_griffin_lim = False

 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
 > Using model: vits
 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > 

In [8]:
# Split translation into English and Thai phrases while preserving order and punctuation
def split_translation(text):
    # Split by spaces but preserve punctuation
    words = text.split()
    
    thai_phrases = []
    english_phrases = []
    phrase_order = [] # Track order of phrases
    
    current_thai = []
    current_english = []
    
    for word in words:
        # Check if word contains any Thai characters
        has_thai = any(ord(c) >= 0x0E00 and ord(c) <= 0x0E7F for c in word)
        
        # Extract any punctuation at end of word
        punctuation = ''
        while word and word[-1] in ',.!?:;':
            punctuation = word[-1] + punctuation
            word = word[:-1]
            
        if has_thai:
            # If we had accumulated English words, add them as a phrase
            if current_english:
                english_phrases.append(' '.join(current_english))
                phrase_order.append(('en', len(english_phrases)-1))
                current_english = []
            current_thai.append(word + punctuation)
        else:
            # If we had accumulated Thai words, add them as a phrase
            if current_thai:
                thai_phrases.append(' '.join(current_thai))
                phrase_order.append(('th', len(thai_phrases)-1))
                current_thai = []
            current_english.append(word + punctuation)
    
    # Add any remaining words
    if current_thai:
        thai_phrases.append(' '.join(current_thai))
        phrase_order.append(('th', len(thai_phrases)-1))
    if current_english:
        english_phrases.append(' '.join(current_english))
        phrase_order.append(('en', len(english_phrases)-1))
            
    return english_phrases, thai_phrases, phrase_order

# Split the translation
english_phrases, thai_phrases, phrase_order = split_translation(translation)

print("English phrases:", english_phrases)
print("Thai phrases:", thai_phrases) 
print("Phrase order:", phrase_order)

English phrases: ['Digital Transformation', 'Microsoft', 'True Digital Park', 'AI', 'Cloud Computing']
Thai phrases: ['เมื่อเดือนที่แล้ว ฉันได้มีโอกาสไปร่วมงานสัมมนา', 'ที่จัดโดย', 'ณ', 'ซึ่งมีวิทยากรจากต่างประเทศมาแชร์ประสบการณ์เกี่ยวกับการใช้', 'และ', 'ในการพัฒนาธุรกิจยุคใหม่']
Phrase order: [('th', 0), ('en', 0), ('th', 1), ('en', 1), ('th', 2), ('en', 2), ('th', 3), ('en', 3), ('th', 4), ('en', 4), ('th', 5)]


# Process reference audio file

In [9]:
# Check if refernec voice is in wav format
current_ref_extension = REFERENCE_PATH.split(".")[-1].lower()

# Convert to wav if not or never converted
if current_ref_extension != "wav":
    if not os.path.exists(REFERENCE_WAV_PATH):
        audio: AudioSegment = AudioSegment.from_file(REFERENCE_PATH, format=current_ref_extension)
        audio.export(REFERENCE_WAV_PATH, format="wav")
    REFERENCE_PATH = REFERENCE_WAV_PATH

In [10]:
# Resmapling reference voice if needed
ref_wav, current_sr = read_audio(REFERENCE_PATH)
if current_sr != C.audio['sample_rate']:
    print('Resampling reference audio...')
    resamapled_ref_wav = resample_wav(ref_wav, current_sr, C.audio['sample_rate'])
    torchaudio.save(REFERENCE_PATH, resamapled_ref_wav[None, :], C.audio['sample_rate'])
else:
    print('This audio is already in the correct sample rate')

This audio is already in the correct sample rate


In [11]:
# trim silence at the beginning and end of the audio
model_and_utils = get_vad_model_and_utils(use_cuda=USE_CUDA, use_onnx=False)

output_path, is_speech = remove_silence(
  model_and_utils,
  REFERENCE_PATH,
  REFERENCE_PATH,
  trim_just_beginning_and_end=True,
  use_cuda=USE_CUDA
)

Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /Users/jackkahod/.cache/torch/hub/master.zip


In [12]:
# normalize the reference audio with rms to -27dB
!ffmpeg-normalize $REFERENCE_PATH -nt rms -t=-27 -o $REFERENCE_PATH -ar 16000 -f

In [13]:
SE_speaker_manager = SpeakerManager(encoder_model_path=C["model_args"]["speaker_encoder_model_path"], encoder_config_path=C["model_args"]["speaker_encoder_config_path"], use_cuda=USE_CUDA)
reference_emb = SE_speaker_manager.compute_embedding_from_clip(REFERENCE_PATH)

 > Model fully restored. 
 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:64
 | > log_func:np.log10
 | > min_level_db:-100
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:20
 | > fft_size:512
 | > power:1.5
 | > preemphasis:0.97
 | > griffin_lim_iters:60
 | > signal_norm:False
 | > symmetric_norm:False
 | > mel_fmin:0
 | > mel_fmax:8000.0
 | > pitch_fmin:1.0
 | > pitch_fmax:640.0
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:4.0
 | > clip_norm:False
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:True
 | > db_level:-27.0
 | > stats_path:None
 | > base:10
 | > hop_length:160
 | > win_length:400


# Inference

In [14]:
# Initialize empty list to store wav segments
wav_segments = []

for (lang, idx) in phrase_order:
    # Select language
    if lang == 'en':
        language_id = 0
        text = english_phrases[idx]
        model.length_scale = 1 # scaler for the duration predictor. The larger it is, the slower the speech.
        model.inference_noise_scale = 0.8 # defines the noise variance applied to the random z vector at inference.
        model.inference_noise_scale_dp = 0.2 # defines the noise variance applied to the duration predictor z vector at inference.
    else:
        language_id = 1
        text = thai_phrases[idx]
        model.length_scale = 1 # scaler for the duration predictor. The larger it is, the slower the speech.
        model.inference_noise_scale = 0.8 # defines the noise variance applied to the random z vector at inference.
        model.inference_noise_scale_dp = 0.2 # defines the noise variance applied to the duration predictor z vector at inference.
    print(f" > text: {text}")
    language_name_to_id = model.language_manager.name_to_id
    language_id_to_name = {v: k for k, v in language_name_to_id.items()}
    print(model.length_scale, model.inference_noise_scale, model.inference_noise_scale_dp)
    print(f"Language ID: {language_id}, Language Name: {language_id_to_name[language_id]}")

    wav, alignment, _, _ = synthesis(
                        model = model,
                        text = text,
                        CONFIG = C,
                        use_cuda = USE_CUDA,
                        d_vector = reference_emb,
                        style_wav = None,
                        language_id = language_id,
                        use_griffin_lim = True,
                        do_trim_silence = False,
                    ).values()
    
    IPython.display.display(Audio(wav, rate=C.audio.sample_rate))
    
    # Append generated wav to segments list
    wav_segments.append(wav)

# Concatenate all wav segments
wav_combined = np.concatenate(wav_segments)
# Display audio
IPython.display.display(Audio(wav_combined, rate=C.audio.sample_rate))

 > text: เมื่อเดือนที่แล้ว ฉันได้มีโอกาสไปร่วมงานสัมมนา
1 0.8 0.2
Language ID: 1, Language Name: th


 > text: Digital Transformation
1 0.8 0.2
Language ID: 0, Language Name: en


 > text: ที่จัดโดย
1 0.8 0.2
Language ID: 1, Language Name: th


 > text: Microsoft
1 0.8 0.2
Language ID: 0, Language Name: en


 > text: ณ
1 0.8 0.2
Language ID: 1, Language Name: th


...

 > text: ซึ่งมีวิทยากรจากต่างประเทศมาแชร์ประสบการณ์เกี่ยวกับการใช้
1 0.8 0.2
Language ID: 1, Language Name: th


 > text: AI
1 0.8 0.2
Language ID: 0, Language Name: en


 > text: และ
1 0.8 0.2
Language ID: 1, Language Name: th


 > text: Cloud Computing
1 0.8 0.2
Language ID: 0, Language Name: en


 > text: ในการพัฒนาธุรกิจยุคใหม่
1 0.8 0.2
Language ID: 1, Language Name: th


In [18]:
language_id = 1
text = translation
model.length_scale = 1 # scaler for the duration predictor. The larger it is, the slower the speech.
model.inference_noise_scale = 0.8 # defines the noise variance applied to the random z vector at inference.
model.inference_noise_scale_dp = 0.8 # defines the noise variance applied to the duration predictor z vector at inference.
print(f" > text: {text}")
language_name_to_id = model.language_manager.name_to_id
language_id_to_name = {v: k for k, v in language_name_to_id.items()}
print(model.length_scale, model.inference_noise_scale, model.inference_noise_scale_dp)
print(f"Language ID: {language_id}, Language Name: {language_id_to_name[language_id]}")

wav, alignment, _, _ = synthesis(
                    model = model,
                    text = text,
                    CONFIG = C,
                    use_cuda = USE_CUDA,
                    d_vector = reference_emb,
                    style_wav = None,
                    language_id = language_id,
                    use_griffin_lim = True,
                    do_trim_silence = False,
                ).values()

Audio(wav, rate=C.audio.sample_rate)

 > text: เมื่อเดือนที่แล้ว ฉันได้มีโอกาสไปร่วมงานสัมมนา Digital Transformation ที่จัดโดย Microsoft ณ True Digital Park ซึ่งมีวิทยากรจากต่างประเทศมาแชร์ประสบการณ์เกี่ยวกับการใช้ AI และ Cloud Computing ในการพัฒนาธุรกิจยุคใหม่
1 0.8 0.8
Language ID: 1, Language Name: th


# Save to Folder

In [16]:
from datetime import datetime

current_time = datetime.now().strftime("%Y_%m_%d_%H_%M_%S")
file_name = model_name + '_' +REFERENCE_FILENAME.split('.')[0] + '_' + current_time + '.wav'
out_path = os.path.join(OUT_PATH, f"{SPEAKER_FOLDER}/{file_name}")

print(f" > Saving output to {out_path}")

os.makedirs(os.path.join(OUT_PATH, SPEAKER_FOLDER), exist_ok=True)
ap.save_wav(wav_combined, out_path)

 > Saving output to output/pipeline/Tsync2_905_mic1/checkpoint_413001_Tsync2_905_mic1_2025_04_13_23_48_59.wav
